# Trabajo Práctico: Segmentación de Departamentos en Arriendo con Clustering

## Magíster en Data Science - Machine Learning II

---

## Objetivo del Trabajo Práctico

En este trabajo práctico aplicaremos técnicas de clustering para segmentar departamentos en arriendo en Estados Unidos. El objetivo es identificar grupos de propiedades con características similares para facilitar su análisis y visualización.

A través de este proceso se busca comprender mejor las distintas categorías de departamentos disponibles en el mercado, considerando variables relevantes del dataset.


---

## Contexto del Problema

Se dispone de un dataset que contiene información sobre departamentos en arriendo en Estados Unidos, incluyendo características como precio, ubicación, tamaño, número de habitaciones, entre otras.

El objetivo del análisis es segmentar los departamentos en grupos homogéneos utilizando técnicas de aprendizaje no supervisado, específicamente clustering. Esto permitirá obtener una representación más clara de los distintos tipos de propiedades disponibles en el mercado.

Para resolver el problema se deberán aplicar las técnicas vistas en clases, realizando un proceso que incluya:

1. Exploración del dataset para identificar las variables disponibles y evaluar cuáles aportan información relevante para el problema.
2. Limpieza de datos, considerando la eliminación de valores nulos, duplicados u otros problemas de calidad de datos si corresponde.
3. Selección de variables adecuadas para realizar la segmentación y elección del modelo de clustering apropiado.
4. Determinación del número óptimo de clusters.
5. Interpretación de los clusters obtenidos, asignándoles nombres representativos que permitan entender fácilmente cada segmento.


---


## Dataset: Apartments for Rent Classified

**Fuente:** Kaggle - Apartments for Rent Classified

**Observaciones:** 200 clientes

**Variables:**

| Variable | Descripción | Tipo |
|----------|-------------|------|
| `id` | Identificador único del apartamento | Numérico (ID) |
| `category` | Categoría del anuncio clasificado | Categórico |
| `title` | Título del anuncio del apartamento | Texto |
| `body` | Descripción detallada del apartamento | Texto |
| `amenities` | Comodidades disponibles (AC, gimnasio, internet, piscina, etc.) | Texto / Categórico |
| `bathrooms` | Número de baños del apartamento | Numérico |
| `bedrooms` | Número de dormitorios del apartamento | Numérico |
| `currency` | Moneda en la que se expresa el precio | Categórico |
| `fee` | Indica si existe una tarifa o comisión adicional | Categórico |
| `has_photo` | Indica si el anuncio incluye foto | Categórico (Sí/No) |
| `pets_allowed` | Tipo de mascotas permitidas (perros, gatos, etc.) | Categórico |
| `price` | Precio de arriendo del apartamento | Numérico |
| `price_display` | Precio formateado para visualización del usuario | Texto |
| `price_type` | Tipo de precio expresado en USD | Categórico |
| `square_feet` | Tamaño del apartamento en pies cuadrados | Numérico |
| `address` | Dirección del apartamento | Texto |
| `cityname` | Ciudad donde se ubica el apartamento | Categórico |
| `state` | Estado o región donde se ubica el apartamento | Categórico |
| `latitude` | Latitud de la ubicación del apartamento | Numérico |
| `longitude` | Longitud de la ubicación del apartamento | Numérico |
| `source` | Fuente de donde proviene el anuncio | Categórico |
| `time` | Fecha y hora de creación del anuncio | Fecha / Tiempo |

## Parte 1: Importación de Librerías y Carga de Datos

Comenzaremos importando las librerías necesarias para nuestro análisis.

In [2]:
# Librerías básicas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Librerías de clustering
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Métricas de evaluación
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Configuración para reproducibilidad
np.random.seed(42)

In [13]:

df = pd.read_csv('apartments_for_rent_classified_100K.csv', sep=';', encoding='latin1')


C:\Users\sergi\AppData\Local\Temp\ipykernel_32020\411995603.py:1: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('apartments_for_rent_classified_100K.csv', sep=';', encoding='latin1')


In [14]:
df.head()

,id,category,title,body,amenities,bathrooms,bedrooms,currency,fee,has_photo,...,price_display,price_type,square_feet,address,cityname,state,latitude,longitude,source,time
0,5668640009,housing/rent/apartment,One BR 507 & 509 Esplanade,"This unit is located at 507 & 509 Esplanade, R...",NaN,1.0,1.0,USD,No,Thumbnail,...,"$2,195",Monthly,542,507 509 Esplanade,Redondo Beach,CA,33.8520,-118.3759,RentLingo,1577360355
1,5668639818,housing/rent/apartment,Three BR 146 Lochview Drive,"This unit is located at 146 Lochview Drive, Ne...",NaN,1.5,3.0,USD,No,Thumbnail,...,"$1,250",Monthly,1500,146 Lochview Dr,Newport News,VA,37.0867,-76.4941,RentLingo,1577360340
2,5668639686,housing/rent/apartment,Three BR 3101 Morningside Drive,This unit is located at 3101 Morningside Drive...,NaN,2.0,3.0,USD,No,Thumbnail,...,"$1,395",Monthly,1650,3101 Morningside Dr,Raleigh,NC,35.8230,-78.6438,RentLingo,1577360332
3,5668639659,housing/rent/apartment,Two BR 209 Aegean Way,"This unit is located at 209 Aegean Way, Vacavi...",NaN,1.0,2.0,USD,No,Thumbnail,...,"$1,600",Monthly,820,209 Aegean Way,Vacaville,CA,38.3622,-121.9712,RentLingo,1577360330
4,5668639374,housing/rent/apartment,One BR 4805 Marquette NE,"This unit is located at 4805 Marquette NE, Alb...",NaN,1.0,1.0,USD,No,Thumbnail,...,$975,Monthly,624,4805 Marquette NE,Albuquerque,NM,35.1038,-106.6110,RentLingo,1577360308


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99492 entries, 0 to 99491
Data columns (total 22 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             99492 non-null  int64  
 1   category       99492 non-null  object 
 2   title          99492 non-null  object 
 3   body           99492 non-null  object 
 4   amenities      83448 non-null  object 
 5   bathrooms      99429 non-null  float64
 6   bedrooms       99368 non-null  float64
 7   currency       99492 non-null  object 
 8   fee            99492 non-null  object 
 9   has_photo      99492 non-null  object 
 10  pets_allowed   39068 non-null  object 
 11  price          99491 non-null  float64
 12  price_display  99491 non-null  object 
 13  price_type     99492 non-null  object 
 14  square_feet    99492 non-null  int64  
 15  address        7943 non-null   object 
 16  cityname       99190 non-null  object 
 17  state          99190 non-null  object 
 18  latitu

In [17]:
# Verificar valores nulos
print("Valores nulos por columna:")
print(df.isnull().sum())
print("\n")

# Verificar duplicados
duplicados = df.duplicated().sum()
print(f"Número de filas duplicadas: {duplicados}")

Valores nulos por columna:
id                   0
category             0
title                0
body                 0
amenities        16044
bathrooms           63
bedrooms           124
currency             0
fee                  0
has_photo            0
pets_allowed     60424
price                1
price_display        1
price_type           0
square_feet          0
address          91549
cityname           302
state              302
latitude            25
longitude           25
source               0
time                 0
dtype: int64


Número de filas duplicadas: 84
